# Options Pricing Lab

Black-Scholes pricing, Greeks (analytical + finite-difference validated), implied volatility solver, and live vol surface from market data.

Per-maturity risk-free rates from the Treasury yield curve, continuous dividend yield, OTM-only filtering, smooth interpolated surface.

In [77]:
import numpy as np
from scipy.stats import norm
from scipy.interpolate import griddata
import pandas as pd
import yfinance as yf
from datetime import datetime, timezone
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'notebook'

In [78]:
# Formulas for European Options
def bs_d1_d2(S, K, T, r, sigma, q=0):
    d1 = (np.log(S/K) + ((r - q + 0.5*sigma**2)*T)) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    return d1, d2

def bs_call_price(S, K, T, r, sigma, q=0):
    d1, d2 = bs_d1_d2(S, K, T, r, sigma, q)
    return S*np.exp(-q*T)*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)

def bs_put_price(S, K, T, r, sigma, q=0):
    d1, d2 = bs_d1_d2(S, K, T, r, sigma, q)
    return K*np.exp(-r*T)*norm.cdf(-d2) - S*np.exp(-q*T)*norm.cdf(-d1)

In [79]:
# Option Greeks (written as option_greek when c_greek = p_greek)
def call_delta(S, K, T, r, sigma, q=0):
    d1, _ = bs_d1_d2(S, K, T, r, sigma, q)
    return norm.cdf(d1)*np.exp(-q*T)

def put_delta(S, K, T, r, sigma, q=0):
    d1, _ = bs_d1_d2(S, K, T, r, sigma, q)
    return -norm.cdf(-d1)*np.exp(-q*T)

def option_gamma(S, K, T, r, sigma, q=0):
    d1, _ = bs_d1_d2(S, K, T, r, sigma, q)
    return np.exp(-q*T)*norm.pdf(d1)/(sigma*np.sqrt(T)*S)

def option_vega(S, K, T, r, sigma, q=0):
    d1, _ = bs_d1_d2(S, K, T, r, sigma, q)
    return S*np.exp(-q*T)*norm.pdf(d1)*np.sqrt(T)

def call_rho(S, K, T, r, sigma, q=0):
    _, d2 = bs_d1_d2(S, K, T, r, sigma, q)
    return T*K*np.exp(-r*T)*norm.cdf(d2)

def put_rho(S, K, T, r, sigma, q=0):
    return call_rho(S, K, T, r, sigma, q) - K*T*np.exp(-r*T)

def call_theta(S, K, T, r, sigma, q=0):
    d1, d2 = bs_d1_d2(S, K, T, r, sigma, q)
    return (-S*np.exp(-q*T)*norm.pdf(d1)*sigma/(2*np.sqrt(T))
            - r*K*np.exp(-r*T)*norm.cdf(d2)
            + q*S*np.exp(-q*T)*norm.cdf(d1))

def put_theta(S, K, T, r, sigma, q=0):
    d1, d2 = bs_d1_d2(S, K, T, r, sigma, q)
    return (-S*np.exp(-q*T)*norm.pdf(d1)*sigma/(2*np.sqrt(T))
            + r*K*np.exp(-r*T)*norm.cdf(-d2)
            - q*S*np.exp(-q*T)*norm.cdf(-d1))

In [80]:
#Finite Difference Greek Validation
e = 1e-5
def fd_call_delta(S, K, T, r, sigma, q=0):
    return (bs_call_price(S+e, K, T, r, sigma, q) - bs_call_price(S-e, K, T, r, sigma, q)) / (2*e)

def fd_put_delta(S, K, T, r, sigma, q=0):
    return (bs_put_price(S+e, K, T, r, sigma, q) - bs_put_price(S-e, K, T, r, sigma, q)) / (2*e)

def fd_option_gamma(S, K, T, r, sigma, q=0):
    p_up = bs_call_price(S+e, K, T, r, sigma, q)
    p_mid = bs_call_price(S, K, T, r, sigma, q)
    p_down = bs_call_price(S-e, K, T, r, sigma, q)
    return (p_up - 2*p_mid + p_down) / (e**2)

def fd_option_vega(S, K, T, r, sigma, q=0):
    return (bs_call_price(S, K, T, r, sigma+e, q) - bs_call_price(S, K, T, r, sigma-e, q)) / (2*e)

def fd_call_rho(S, K, T, r, sigma, q=0):
    return (bs_call_price(S, K, T, r+e, sigma, q) - bs_call_price(S, K, T, r-e, sigma, q)) / (2*e)

def fd_put_rho(S, K, T, r, sigma, q=0):
    return (bs_put_price(S, K, T, r+e, sigma, q) - bs_put_price(S, K, T, r-e, sigma, q)) / (2*e)

def fd_call_theta(S, K, T, r, sigma, q=0):
    return -(bs_call_price(S, K, T+e, r, sigma, q) - bs_call_price(S, K, T-e, r, sigma, q)) / (2*e)

def fd_put_theta(S, K, T, r, sigma, q=0):
    return -(bs_put_price(S, K, T+e, r, sigma, q) - bs_put_price(S, K, T-e, r, sigma, q)) / (2*e)

In [81]:
#Implied Vol Solver
def implied_volatility(market_price, S, K, T, r, option_type, q=0,
                       initial_guess=0.2, tolerance=1e-8, max_iter=100):
    sigma = initial_guess
    pricer = bs_call_price if option_type == 'call' else bs_put_price
    for i in range(max_iter):
        price = pricer(S, K, T, r, sigma, q)
        residual = price - market_price
        if np.abs(residual) < tolerance:
            return sigma
        vega = option_vega(S, K, T, r, sigma, q)
        sigma = sigma - residual / vega
    return None

In [82]:
def build_yield_curve():
    """Pull a few points on the Treasury curve. Returns {tenor_years: rate_decimal}."""
    tenors = {'^IRX': 0.25, '^FVX': 5.0, '^TNX': 10.0, '^TYX': 30.0}
    curve = {}
    for tk, t in tenors.items():
        try:
            y = yf.Ticker(tk).history(period="5d")['Close'].iloc[-1] / 100
            curve[t] = y
        except Exception as ex:
            print(f"  failed to fetch {tk}: {ex}")
    return curve

def rate_for_maturity(T, curve):
    """Linear interpolation in the yield curve. Clamps at endpoints."""
    tenors = sorted(curve.keys())
    yields = [curve[t] for t in tenors]
    return float(np.interp(T, tenors, yields))

def get_dividend_yield(ticker_symbol):
    """Trailing 12-month dividend yield as a decimal."""
    tk = yf.Ticker(ticker_symbol)
    info_yield = tk.info.get('dividendYield')
    if info_yield is not None and info_yield > 0:
        return info_yield / 100 if info_yield > 1 else info_yield
    divs = tk.dividends
    if len(divs) == 0:
        return 0.0
    one_year_ago = datetime.now(timezone.utc) - pd.Timedelta(days=365)
    if divs.index.tz is None:
        divs.index = divs.index.tz_localize('UTC')
    recent = divs[divs.index > one_year_ago].sum()
    spot = tk.history(period="1d")['Close'].iloc[-1]
    return float(recent / spot)

def fetch_chain(ticker_symbol):
    """Pull full options chain for all available expirations."""
    ticker = yf.Ticker(ticker_symbol)
    spot = ticker.history(period="1d")['Close'].iloc[-1]
    rows = []
    for exp_str in ticker.options:
        try:
            chain = ticker.option_chain(exp_str)
        except Exception as ex:
            print(f"  failed to fetch {exp_str}: {ex}")
            continue
        for option_type, df in [('call', chain.calls), ('put', chain.puts)]:
            for _, row in df.iterrows():
                rows.append({
                    'expiration': exp_str,
                    'type': option_type,
                    'strike': row['strike'],
                    'bid': row['bid'],
                    'ask': row['ask'],
                    'last': row['lastPrice'],
                    'volume': row['volume'],
                    'open_interest': row['openInterest'],
                    'yf_iv': row['impliedVolatility'],
                })
    df = pd.DataFrame(rows)
    df['expiration'] = pd.to_datetime(df['expiration'])
    now = datetime.now()
    df['T'] = (df['expiration'] - now).dt.total_seconds() / (365.25 * 86400)
    df['spot'] = spot
    return df

In [83]:
def filter_chain(df, spot, max_moneyness=0.5, min_volume=10,
                 min_open_interest=100, max_rel_spread=0.20):
    df = df.copy()
    df['mid'] = (df['bid'] + df['ask']) / 2
    df['rel_spread'] = (df['ask'] - df['bid']) / df['mid'].replace(0, float('nan'))
    df['moneyness'] = df['strike'] / spot
    mask = (
        (df['bid'] > 0) &
        (df['ask'] > 0) &
        (df['mid'] > 0.05) &
        (df['volume'].fillna(0) >= min_volume) &
        (df['open_interest'].fillna(0) >= min_open_interest) &
        (df['rel_spread'] < max_rel_spread) &
        (df['moneyness'] > 1 - max_moneyness) &
        (df['moneyness'] < 1 + max_moneyness) &
        (df['T'] > 7/365) &
        # OTM-only: calls where K >= S, puts where K <= S
        (((df['type'] == 'call') & (df['strike'] >= spot)) |
         ((df['type'] == 'put') & (df['strike'] <= spot)))
    )
    return df[mask].copy()

def compute_ivs(df, yield_curve, q=0):
    """Apply IV solver to every row using per-maturity rate."""
    df = df.copy()
    ivs = []
    failures = 0
    for _, row in df.iterrows():
        r_for_this = rate_for_maturity(row['T'], yield_curve)
        try:
            iv = implied_volatility(
                market_price=row['mid'],
                S=row['spot'],
                K=row['strike'],
                T=row['T'],
                r=r_for_this,
                option_type=row['type'],
                q=q,
                tolerance=1e-8,
            )
        except Exception:
            iv = None
        if iv is None or iv <= 0 or iv > 5.0:
            iv = None
            failures += 1
        ivs.append(iv)
    df['iv'] = ivs
    print(f"  IV computed for {df['iv'].notna().sum()} of {len(df)} options ({failures} failed)")
    return df.dropna(subset=['iv'])

In [84]:
def build_surface(ticker_symbol):
    print(f"Building surface for {ticker_symbol}...")
    chain_df = fetch_chain(ticker_symbol)
    spot = chain_df['spot'].iloc[0]
    yield_curve = build_yield_curve()
    q = get_dividend_yield(ticker_symbol)
    print(f"  Spot: ${spot:.2f}")
    print(f"  Dividend yield: {q:.4f}")
    print(f"  Raw options: {len(chain_df)}")
    clean = filter_chain(chain_df, spot)
    print(f"  After filtering: {len(clean)}")
    clean_with_iv = compute_ivs(clean, yield_curve, q=q)
    return clean_with_iv, q, spot

def render_surface(df,ticker_symbol,spot):
    """Plot the IV surface — both calls and puts, since both are OTM."""
    df = df.copy()
    df['moneyness'] = df['strike'] / df['spot']
    df['days'] = df['T'] * 365
    mny = df['moneyness'].values
    days = df['days'].values
    iv = df['iv'].values * 100
    mny_grid = np.linspace(mny.min(), mny.max(), 60)
    days_grid = np.linspace(days.min(), days.max(), 60)
    MNY, DAYS = np.meshgrid(mny_grid, days_grid)
    IV = griddata((mny, days), iv, (MNY, DAYS), method='linear')
    fig = go.Figure()
    fig.add_trace(go.Surface(
        x=MNY, y=DAYS, z=IV,
        colorscale='Viridis',
        colorbar=dict(title='IV (%)'),
        hovertemplate=('Moneyness: %{x:.3f}<br>Days: %{y:.0f}<br>IV: %{z:.1f}%<extra></extra>'),
        opacity=0.95,
    ))
    fig.add_trace(go.Scatter3d(
        x=mny, y=days, z=iv,
        mode='markers',
        marker=dict(size=2, color='white', opacity=0.4),
        name='Market quotes',
        hovertemplate=('Strike: $%{customdata[0]:.0f}<br>Days: %{y:.0f}<br>IV: %{z:.1f}%<extra></extra>'),
        customdata=df[['strike']].values,
    ))
    fig.update_layout(
        title=f"{ticker_symbol} Implied Volatility Surface — spot ${spot:.2f}, {datetime.now():%Y-%m-%d}",
        scene=dict(
            xaxis_title='Moneyness (K/S)',
            yaxis_title='Days to expiry',
            zaxis_title='Implied vol (%)',
            camera=dict(eye=dict(x=1.6, y=-1.6, z=1.2)),
        ),
        width=1100, height=700,
    )
    fig.show()

In [95]:
def plot_smile(df, days_target=30):
    df = df.copy()
    df['days'] = df['T'] * 365
    df['moneyness'] = df['strike'] / df['spot']    
    unique_days = df['days'].unique()
    chosen = unique_days[np.argmin(np.abs(unique_days - days_target))]   
    smile = df[df['days'] == chosen].sort_values('moneyness')    
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=smile['moneyness'], y=smile['iv'] * 100,
        mode='markers+lines', marker=dict(size=6),
        name=f'{chosen:.0f} days',
    ))
    fig.add_vline(x=1.0, line_dash='dash', line_color='red', annotation_text='ATM')
    fig.update_layout(
        title=f'Volatility smile, {TICKER}, T={chosen:.0f} days',
        xaxis_title='Moneyness (K/S)',
        yaxis_title='Implied volatility (%)',
        width=800, height=500,
    )
    fig.show()

In [97]:
TICKER = "SPY"

df, q, spot = build_surface(TICKER)
render_surface(df, TICKER, spot)

Building surface for SPY...
  Spot: $741.25
  Dividend yield: 0.0103
  Raw options: 9847
  After filtering: 1836
  IV computed for 1684 of 1836 options (152 failed)


In [98]:
#Easier vol smile plot
plot_smile(df)